In [1]:
# rm(list = ls())
# gc()
library(MASS) # 多元正态分布
library(Rcpp) # R链接c++语言
# library(ggplot2) # 绘图
# library(reshape2) # 数据融合
library(compiler)
library(survival)
# 平行计算
library(foreach)
library(iterators)
library(parallel)
library(doParallel) # 仅此一个也行
sourceCpp('TestVCM.cpp')
# sourceCpp('TestCPLRM3.cpp')
# 矩阵计算函数(未除以n): angle_matrix()
# 带Bootstrap的统计量仿真函数: testCPLRM()
r_files <- list.files('speedyCMI/')
for (file in r_files) {
    source(paste0('speedyCMI/', file))    
}

R.version

               _                           
platform       x86_64-pc-linux-gnu         
arch           x86_64                      
os             linux-gnu                   
system         x86_64, linux-gnu           
status                                     
major          3                           
minor          5.2                         
year           2018                        
month          12                          
day            20                          
svn rev        75870                       
language       R                           
version.string R version 3.5.2 (2018-12-20)
nickname       Eggshell Igloo              

In [2]:
data = read.csv('PBC_data/cirrhosis2_2.csv', encoding = 'UTF-8')
# remove missing value
# del_names <- c('Chol', 'Trig', 'Chol', 'Copper', 'Platelets')
# for (i in del_names){
#     data = data[data[i] != '.',]
# }

data = data[data$Status != 1, ] # 19(18) = 11 + 8 dead with liver transplant
data$Status[data$Status == 2] = 1

col_names <- colnames(data)
var_names <- setdiff(col_names, c('ID', 'Futime', 'Status', 'Drug', 'Chol', 'Alk_Phos'))

# df1

In [3]:
i = 1
df_drug1 <- data[data$Drug == i, ]
# df_drug1 <- df_drug1[-c(23, 95), ]
stad_var_names <- c('Alk_Phos', 'Age', 'Bili', 'Copper', 'Trig', 'Platelets', 'Prothrombin', 'Chol')
# for (j in stad_var_names) {
#     df_drug1[j] <- scale(df_drug1[j])
# }

Y = log(as.numeric(df_drug1$Alk_Phos))
X = as.matrix(df_drug1[var_names])
T1 = as.numeric(df_drug1$Futime)
U <- as.numeric(df_drug1$Chol)
# 标准化
# Y <- scale(Y)
# X <- apply(X, 2, FUN = function(x) {scale(x)})
# T1 <- scale(T1)
# U <- scale(U)


delta = as.numeric(df_drug1$Status)
dat = data.frame(z = X[, 1], w = T1, y = Y, d = delta)
imp_dat1 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "exponential",
                          data = dat)
imp_dat2 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "weibull",
                          data = dat)

print(c('censoring rate:', round(1 - mean(delta), 4)))
print(c('sample size:', length(Y)))

[1] "censoring rate:" "0.5512"         
[1] "sample size:" "127"         


In [4]:
method <- "Test_VCM"
h_con <- 1.54 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5512"         
[1] "sample size:" "127"         
带宽: 1.54 	 p-值: 


0.4545,0.886,0.6585,0.446,0.2205,0.285,0.1275,0.8095,0.8595


In [5]:
method <- "Test_VCM"
h_con <- 3.14 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5512"         
[1] "sample size:" "127"         
带宽: 3.14 	 p-值: 


0.062,0.605,0.0515,0.0545,0.2405,0.283,0.617,0.6715,0.6395


In [6]:
method <- "Test_VCM"
h_con <- 2.34 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5512"         
[1] "sample size:" "127"         
带宽: 2.34 	 p-值: 


0.4525,0.976,0.305,0.443,0.1755,0.237,0.0935,0.9745,0.976


In [7]:
# i = 1
# df_drug1 <- data[data$Drug == i, ]
# df_drug1 <- df_drug1[-c(23, 95), ]
# Y = as.numeric(df_drug1$Alk_Phos)
# X = data.matrix(df_drug1[var_names])
# T1 = as.numeric(df_drug1$Futime)
# U <- as.numeric(df_drug1$Chol)
# delta = as.numeric(df_drug1$Status)

# Y <- scale(Y)
# X <- apply(X, 2, FUN = function(x) {scale(x)})
# T1 <- scale(T1)
# U <- scale(U)

In [8]:
h_con <- 2.34
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = 1, h_con = h_con, m = m)
res111 <- do.call(method, Data_drug1)

In [9]:
n2 <- length(Y)
p2 <- dim(X)[2]
m <- 6

yhats2 <- t(matrix(rep(1, n2, each = m), m))
errors2 <- t(matrix(rep(1, n2, each = m), m))


yhats2[, 1] <- res111$yhat1
errors2[, 1] <- res111$resid1w

yhats2[, 2] <- c(res111$yhat2, rep(0, n2 - length(res111$yhat2)))
errors2[, 2] <- c(res111$resid2, rep(0, n2 - length(res111$resid2)))

yhats2[, 3] <- res111$yhat3
errors2[, 3] <- res111$resid3

yhats2[, 4] <- res111$yhat4
errors2[, 4] <- res111$resid4

yhats2[, 5] <- res111$yhat5
errors2[, 5] <- res111$resid5

yhats2[, 6] <- res111$yhat6
errors2[, 6] <- res111$resid6



write.table(yhats2, file = 'results/yhats1_1_log.txt', row.names = FALSE, col.names = FALSE)
write.table(errors2, file = 'results/errors1_1_log.txt', row.names = FALSE, col.names = FALSE)


# df2

In [11]:
i = 2
df_drug1 <- data[data$Drug == i, ]
# df_drug1 <- df_drug1[-c(23, 95), ]
stad_var_names <- c('Alk_Phos', 'Age', 'Bili', 'Copper', 'Trig', 'Platelets', 'Prothrombin', 'Chol')
# for (j in stad_var_names) {
#     df_drug1[j] <- scale(df_drug1[j])
# }

Y = log(as.numeric(df_drug1$Alk_Phos))
X = as.matrix(df_drug1[var_names])
T1 = as.numeric(df_drug1$Futime)
U <- as.numeric(df_drug1$Chol)
# 标准化
# Y <- scale(Y)
# X <- apply(X, 2, FUN = function(x) {scale(x)})
# T1 <- scale(T1)
# U <- scale(U)


delta = as.numeric(df_drug1$Status)
dat = data.frame(z = X[, 1], w = T1, y = Y, d = delta)
imp_dat1 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "exponential",
                          data = dat)
imp_dat2 = cmi_fp_analytical(imputation_model = Surv(time = w, event = d) ~ z,
                          dist = "weibull",
                          data = dat)

print(c('censoring rate:', round(1 - mean(delta), 4)))
print(c('sample size:', length(Y)))

[1] "censoring rate:" "0.5878"         
[1] "sample size:" "131"         


In [12]:
method <- "Test_VCM"
h_con <- 1.54 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5878"         
[1] "sample size:" "131"         
带宽: 1.54 	 p-值: 


0.18,0.255,0.172,0.2015,0.026,0.033,0.123,0.268,0.243


In [13]:
method <- "Test_VCM"
h_con <- 3.14 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5878"         
[1] "sample size:" "131"         
带宽: 3.14 	 p-值: 


0.065,0.676,0.025,0.062,0.1835,0.192,0.4005,0.792,0.718


In [14]:
method <- "Test_VCM"
h_con <- 2.34 # 带宽系数
# n_con <- 3
m <- 50
B <- 2000
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = B, h_con = h_con, m = m)

print(c('censoring rate:', round(1 - mean(Data_drug1$delta), 4)))
print(c('sample size:', length(Data_drug1$Y)))

res <- do.call(method, Data_drug1)
cat("带宽:", h_con, '\t', "p-值:", '\n')
res$p_value

[1] "censoring rate:" "0.5878"         
[1] "sample size:" "131"         
带宽: 2.34 	 p-值: 


0.1305,0.5325,0.0895,0.1405,0.416,0.4015,0.197,0.6305,0.525


In [15]:
h_con <- 2.34
Data_drug1 <- list(Y = Y, X = X, T1 = T1, delta = delta, U = U, V = T1,
                   VE = imp_dat1$imputed_data$imp, VW = imp_dat2$imputed_data$imp,
                   B = 1, h_con = h_con, m = m)
res112 <- do.call(method, Data_drug1)

In [16]:
n2 <- length(Y)
p2 <- dim(X)[2]
m <- 6

yhats2 <- t(matrix(rep(1, n2, each = m), m))
errors2 <- t(matrix(rep(1, n2, each = m), m))


yhats2[, 1] <- res112$yhat1
errors2[, 1] <- res112$resid1w

yhats2[, 2] <- c(res112$yhat2, rep(0, n2 - length(res112$yhat2)))
errors2[, 2] <- c(res112$resid2, rep(0, n2 - length(res112$resid2)))

yhats2[, 3] <- res112$yhat3
errors2[, 3] <- res112$resid3

yhats2[, 4] <- res112$yhat4
errors2[, 4] <- res112$resid4

yhats2[, 5] <- res112$yhat5
errors2[, 5] <- res112$resid5

yhats2[, 6] <- res112$yhat6
errors2[, 6] <- res112$resid6



write.table(yhats2, file = 'results/yhats1_2_log.txt', row.names = FALSE, col.names = FALSE)
write.table(errors2, file = 'results/errors1_2_log.txt', row.names = FALSE, col.names = FALSE)
